<a href="https://colab.research.google.com/github/2024meb1326-eng/Stress_Prediction-/blob/main/code/Ques_8_A).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Input, BatchNormalization, Dropout
from keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(53)
tf.random.set_seed(53)

initializer1 = keras.initializers.HeNormal()

print("Loading output.xlsx ...")
df = pd.read_excel("output.xlsx", header=0)
X = df.values.astype(np.float32)
print("X shape:", X.shape)


print("Loading stress files ...")
stress_files = sorted(
    glob.glob(os.path.join("stress", "*.txt")),
    key=lambda p: int(os.path.splitext(os.path.basename(p))[0].split("_")[1])
)

Y = np.zeros((len(X), 1), dtype=np.float32)
for i, fpath in enumerate(stress_files):
    data = np.loadtxt(fpath)
    Y[i, 0] = data[:, 0].max()
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(stress_files)} files loaded ...")

print("Y shape:", Y.shape)
print(f"Y  min={Y.min():.4f}  max={Y.max():.4f}  mean={Y.mean():.4f}")

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)


pca = PCA(n_components=0.95, random_state=53)
X_pca = pca.fit_transform(X_scaled)
print(f"\nPCA: reduced {X_scaled.shape[1]} → {X_pca.shape[1]} components (95% variance retained)")


Y_log = np.log1p(Y)
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y_log)

X_train, X_test, Y_train, Y_test = train_test_split(X_pca, Y_scaled, test_size=0.2, random_state=40)
print(f"Train size: {X_train.shape[0]}   Test size: {X_test.shape[0]}")
print(f"Input dim after PCA: {X_train.shape[1]}")


tf.keras.backend.clear_session()

n_inputs = X_train.shape[1]

model1 = Sequential()
model1.add(Input(shape=(n_inputs,), name='Input_Layer'))

model1.add(Dense(64, kernel_initializer=initializer1, kernel_regularizer=l2(5e-2), name='Hidden1'))
model1.add(keras.layers.Activation('relu', name='Relu1'))
model1.add(Dropout(0.1, name='Drop1'))
model1.add(Dense(32, kernel_initializer=initializer1, kernel_regularizer=l2(5e-2), name='Hidden2'))
model1.add(keras.layers.Activation('relu', name='Relu2'))
model1.add(Dropout(0.05 , name='Drop2'))

model1.add(Dense(16, kernel_initializer=initializer1, kernel_regularizer=l2(5e-2), name='Hidden3'))
model1.add(keras.layers.Activation('relu', name='Relu3'))


model1.add(Dense(1, name='Output'))

model1.summary()

model1.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr  = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)

history = model1.fit(X_train, Y_train, epochs=200, batch_size=64,
                     validation_split=0.2, callbacks=[early_stop, reduce_lr])

Y_pred_test_scaled  = model1.predict(X_test)
Y_pred_train_scaled = model1.predict(X_train)

Y_pred_test_log  = scaler_Y.inverse_transform(Y_pred_test_scaled)
Y_pred_train_log = scaler_Y.inverse_transform(Y_pred_train_scaled)
Y_test_log       = scaler_Y.inverse_transform(Y_test)
Y_train_log      = scaler_Y.inverse_transform(Y_train)

Y_pred_test  = np.expm1(Y_pred_test_log).ravel()
Y_pred_train = np.expm1(Y_pred_train_log).ravel()
Y_test_orig  = np.expm1(Y_test_log).ravel()
Y_train_orig = np.expm1(Y_train_log).ravel()

r2_test   = r2_score(Y_test_orig, Y_pred_test)
mae_test  = mean_absolute_error(Y_test_orig, Y_pred_test)
rmse_test = np.sqrt(mean_squared_error(Y_test_orig, Y_pred_test))
mape_test = np.mean(np.abs((Y_test_orig - Y_pred_test) / (Y_test_orig + 1e-8))) * 100

r2_train   = r2_score(Y_train_orig, Y_pred_train)
mae_train  = mean_absolute_error(Y_train_orig, Y_pred_train)
rmse_train = np.sqrt(mean_squared_error(Y_train_orig, Y_pred_train))
mape_train = np.mean(np.abs((Y_train_orig - Y_pred_train) / (Y_train_orig + 1e-8))) * 100

print("\n========== TEST SET METRICS ==========")
print(f"R²   : {r2_test:.5f}")
print(f"MAE  : {mae_test:.5f}")
print(f"RMSE : {rmse_test:.5f}")
print(f"MAPE : {mape_test:.3f} %")

print("\n========== TRAIN SET METRICS ==========")
print(f"R²   : {r2_train:.5f}")
print(f"MAE  : {mae_train:.5f}")
print(f"RMSE : {rmse_train:.5f}")
print(f"MAPE : {mape_train:.3f} %")

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training & Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Training & Validation MAE')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

lo_test = min(Y_test_orig.min(), Y_pred_test.min()) * 0.97
hi_test = max(Y_test_orig.max(), Y_pred_test.max()) * 1.03

plt.figure(figsize=(6, 6))
plt.scatter(Y_test_orig, Y_pred_test, s=15, alpha=0.6, color='steelblue', label='Test samples')
plt.plot([lo_test, hi_test], [lo_test, hi_test], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Max Von Mises Stress')
plt.ylabel('Predicted Max Von Mises Stress')
plt.title(f'Parity Plot - Test Set\nR²={r2_test:.4f}  RMSE={rmse_test:.4f}  MAPE={mape_test:.2f}%')
plt.legend()
plt.tight_layout()
plt.savefig('parity_test.png', dpi=150)
plt.show()

lo_train = min(Y_train_orig.min(), Y_pred_train.min()) * 0.97
hi_train = max(Y_train_orig.max(), Y_pred_train.max()) * 1.03

plt.figure(figsize=(6, 6))
plt.scatter(Y_train_orig, Y_pred_train, s=10, alpha=0.4, color='darkorange', label='Train samples')
plt.plot([lo_train, hi_train], [lo_train, hi_train], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Max Von Mises Stress')
plt.ylabel('Predicted Max Von Mises Stress')
plt.title(f'Parity Plot - Train Set\nR²={r2_train:.4f}  RMSE={rmse_train:.4f}  MAPE={mape_train:.2f}%')
plt.legend()
plt.tight_layout()
plt.savefig('parity_train.png', dpi=150)
plt.show()

residuals = Y_test_orig - Y_pred_test

plt.figure(figsize=(7, 5))
plt.scatter(Y_pred_test, residuals, s=15, alpha=0.6, color='steelblue')
plt.axhline(0, color='red', lw=2, ls='--', label='Zero residual')
plt.xlabel('Predicted Max Von Mises Stress')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot - Test Set')
plt.legend()
plt.tight_layout()
plt.savefig('residual_plot.png', dpi=150)
plt.show()

print("\nAll plots saved.")